# Minimal Baseline — Qwen3-VL 4B + LoRA (VQA, 4지선다)

이미지 + 질문 + 보기(a~d)를 입력받아 정답 한 글자를 고르는 VQA baseline입니다.

**전체 흐름**
1. 튜닝 파라미터 설정 (실험 시 이 셀만 수정)
2. 경로 / 환경 변수 설정
3. 라이브러리 import & 시드 고정
4. 데이터 준비 (zip 해제 → 고정 split → CSV 로드)
5. 모델 로드 & LoRA 부착
6. 프롬프트 / Dataset / Collator 구성
7. 학습
8. 추론 함수 정의
9. Validation 평가
10. Test 추론 & 제출 파일 생성

**baseline 설계**
- validation: `data/split/validation.csv` 고정 분할 사용
- LoRA: Vision + Attention 학습, MLP LoRA 제외
- loss: 정답 `a/b/c/d` 토큰 위치만 학습 (label smoothing 없음)
- 추론: 생성(generate) 대신 마지막 위치의 a/b/c/d 로짓만 비교해 argmax
- 1 epoch 학습 후 validation 1회 평가
- 로깅: W&B (`USE_WANDB`로 on/off) — train loss/LR은 Trainer가, validation accuracy는 직접 기록

## 1. 튜닝 파라미터
성능에 영향을 주는 값은 모두 이 셀에 모았습니다. 실험할 때는 이 셀만 수정하고 전체 재실행하면 됩니다.

In [ ]:
# =====================================================================
# 튜닝 파라미터 — 실험 시 이 셀만 수정
# =====================================================================

# ---------------------------------------------------------------------
# [모델]
# ---------------------------------------------------------------------
# Unsloth가 미리 4bit(bnb) 양자화해 둔 Qwen3-VL 4B Instruct.
# 4bit 베이스 + LoRA = QLoRA 구조. 더 큰 모델로 바꾸면 성능↑ 가능성, VRAM/시간↑.
MODEL_NAME = "unsloth/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit"
SEED = 42  # 데이터 셔플, LoRA 초기화, 학습 전반의 재현성용

# ---------------------------------------------------------------------
# [이미지 해상도]
# ---------------------------------------------------------------------
# 비전 인코더는 이미지를 패치 단위로 잘라 토큰화함.
# 이미지 픽셀 수를 [MIN_PIXELS, MAX_PIXELS] 범위로 리사이즈 → 비전 토큰 수가 이 범위로 제한됨.
#   - 값을 키우면: 작은 글자/세부 객체 인식↑, 메모리·학습시간↑
#   - 값을 줄이면: 속도↑, 세부 정보 손실
PATCH_PIXELS = 32 * 32            # 비전 토큰 1개가 담당하는 픽셀 수 (Qwen3-VL: patch 16 × spatial merge 2)
MIN_PIXELS = 256 * PATCH_PIXELS   # 최소 약 256 비전 토큰
MAX_PIXELS = 1280 * PATCH_PIXELS  # 최대 약 1280 비전 토큰

# 이미지 토큰 + 텍스트 토큰 전체 최대 길이.
# MAX_PIXELS를 올리면 이 값도 같이 올려야 잘림(truncation)이 발생하지 않음.
MAX_SEQ_LEN = 2048

# ---------------------------------------------------------------------
# [LoRA]
# ---------------------------------------------------------------------
LORA_R = 16          # LoRA rank. 클수록 표현력↑, 파라미터 수·과적합 위험↑ (8/16/32/64 등)
LORA_ALPHA = 32      # 스케일 = alpha / r. 보통 r 또는 2r로 설정
LORA_DROPOUT = 0.0   # LoRA 경로 dropout. 0이면 Unsloth 최적화 커널 사용 가능

# 어느 모듈에 LoRA를 붙일지
FINETUNE_VISION = False     # 비전 인코더 레이어
FINETUNE_LANGUAGE = True   # 언어 모델 레이어
FINETUNE_ATTENTION = True  # q/k/v/o projection
FINETUNE_MLP = True       # gate/up/down projection (baseline에서는 제외)

# ---------------------------------------------------------------------
# [학습]
# ---------------------------------------------------------------------
EPOCHS = 1.0            # 학습 epoch 수 (float 가능: 0.5 = 절반만 학습)
LR = 1e-4               # 학습률. LoRA는 보통 5e-5 ~ 2e-4 범위에서 탐색
BATCH_SIZE = 8          # GPU당 배치 크기 (VRAM 부족 시 줄이고 GRAD_ACCUM을 늘릴 것)
GRAD_ACCUM = 2          # gradient accumulation. 실효 배치 = BATCH_SIZE * GRAD_ACCUM
LR_SCHEDULER = "cosine" # 학습률 스케줄 ("linear", "cosine", "constant" 등)
WARMUP_RATIO = 0.05     # 전체 step 중 워밍업 비율
WEIGHT_DECAY = 0.01     # L2 정규화 강도
OPTIM = "adamw_8bit"    # 8bit AdamW: 옵티마이저 상태 메모리 절약
MAX_GRAD_NORM = 1.0     # gradient clipping 임계값
LOGGING_STEPS = 10      # 몇 step마다 train loss를 로깅할지

# ---------------------------------------------------------------------
# [프롬프트]
# ---------------------------------------------------------------------
# 시스템 프롬프트: 모델의 역할과 출력 형식을 지정
SYSTEM_PROMPT = (
    "당신은 이미지를 보고 질문에 답하는 시각 질의응답 도우미입니다. "
    "a, b, c, d 중 정확히 한 글자로만 답하고, 설명은 하지 마세요."
)
# 질문/보기 뒤에 붙는 출력 형식 지시문
USER_INSTRUCTION = "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."

# ---------------------------------------------------------------------
# [추론]
# ---------------------------------------------------------------------
INFER_BATCH_SIZE = 4  # 추론 배치 크기 (정확도 영향 없음, 속도/메모리만 영향)

# ---------------------------------------------------------------------
# [로깅 - W&B]
# ---------------------------------------------------------------------
USE_WANDB = True                 # False면 W&B 없이 실행
WANDB_PROJECT = "stv"
WANDB_RUN_NAME = "baseline-minimal"

## 2. 경로 / 환경 변수
RunPod 여부를 자동 감지해 데이터·캐시·출력 경로를 결정합니다. 환경 변수(`STV_*`)로 덮어쓸 수 있습니다.

`HF_HOME` 등 환경 변수는 **unsloth/transformers import 이전에** 설정해야 적용되므로 import 셀보다 먼저 실행합니다.

In [ ]:
from pathlib import Path
import os, sys, zipfile

# RunPod 컨테이너인지 판별: RUNPOD_POD_ID 환경 변수 또는 /workspace 디렉토리 존재 여부
ON_RUNPOD = bool(os.environ.get("RUNPOD_POD_ID")) or Path("/workspace").is_dir()

# 레포 루트 탐색: 현재 디렉토리부터 상위로 올라가며 eda/split_validation.py가 있는 폴더를 찾음
REPO_ROOT = next(
    (d for d in [Path.cwd(), *Path.cwd().parents] if (d / "eda" / "split_validation.py").exists()),
    Path.cwd(),
)

# ---- 데이터 경로 ----
# 원본 데이터 zip 위치
ZIP_PATH = Path(os.environ.get(
    "STV_DATA_ZIP",
    "/workspace/stv/data.zip" if ON_RUNPOD else str(REPO_ROOT / "data.zip"),
))
# 압축 해제 및 split 저장 루트 (RunPod에서는 빠른 로컬 디스크 /root 사용)
DATA_ROOT = Path(os.environ.get(
    "STV_DATA_ROOT",
    "/root/data" if ON_RUNPOD else str(REPO_ROOT / "data"),
))
RAW_BASE_DIR = DATA_ROOT / "raw"    # zip 압축 해제 위치
SPLIT_DIR = DATA_ROOT / "split"     # 고정 train/validation split CSV 위치

# ---- 출력 경로 (LoRA 가중치, submission 저장) ----
OUTPUT_DIR = Path(os.environ.get(
    "STV_OUTPUT_DIR",
    "/workspace/stv/outputs/baseline" if ON_RUNPOD else str(REPO_ROOT / "output"),
))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- 라이브러리 환경 변수 (import 전에 설정해야 함) ----
# HuggingFace 모델 캐시 위치: RunPod에서는 영속 볼륨(/workspace)에 두어 재다운로드 방지
os.environ.setdefault("HF_HOME", "/workspace/.cache/huggingface" if ON_RUNPOD else str(REPO_ROOT / ".hf-cache"))
os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"  # Unsloth 사용 통계 전송 비활성화
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # DataLoader worker와 tokenizer 병렬화 충돌 경고 방지

# 정답 후보 (고정 상수)
CHOICES = ["a", "b", "c", "d"]

print("repo  :", REPO_ROOT)
print("zip   :", ZIP_PATH)
print("output:", OUTPUT_DIR)

## 3. 라이브러리 import & 시드 고정
`unsloth`는 transformers/trl을 패치하므로 **다른 라이브러리보다 먼저** import합니다.

In [ ]:
# unsloth를 가장 먼저 import해야 transformers/trl 패치가 정상 적용됨
import unsloth
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator

import random
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from trl import SFTTrainer, SFTConfig

if USE_WANDB:
    import wandb

assert torch.cuda.is_available(), "CUDA GPU가 필요합니다."

# ---- 재현성을 위한 시드 고정 ----
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ---- 연산 설정 ----
# TF32: Ampere 이상 GPU에서 fp32 행렬곱을 빠르게 수행 (정밀도 손실 미미)
torch.backends.cuda.matmul.allow_tf32 = True
if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.allow_tf32 = True

# 매우 큰 이미지 로드 시 PIL의 DecompressionBomb 경고/에러 방지
Image.MAX_IMAGE_PIXELS = None

print("GPU:", torch.cuda.get_device_name())

## 4. 데이터 준비
1. `data.zip` 압축 해제 (이미 해제되어 있으면 건너뜀)
2. `eda/split_validation.py`의 `ensure_split`으로 고정 train/validation split 생성 또는 재사용
3. CSV 로드 및 train/valid 중복 검사

CSV 컬럼: `id`, `path`(이미지 상대경로), `question`, `a`~`d`(보기), `answer`(train/valid만)

In [ ]:
RAW_BASE_DIR.mkdir(parents=True, exist_ok=True)

# ---- 1) zip 압축 해제 ----
# train.csv가 이미 있으면 해제된 것으로 간주하고 건너뜀
if not list(RAW_BASE_DIR.rglob("train.csv")):
    if not ZIP_PATH.exists():
        raise FileNotFoundError(f"dataset zip not found: {ZIP_PATH}")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(RAW_BASE_DIR)

# zip 내부 폴더 구조와 무관하게 train.csv가 있는 폴더를 원본 데이터 폴더로 사용
train_csv = list(RAW_BASE_DIR.rglob("train.csv"))[0]
RAW_DATA_DIR = train_csv.parent  # 이미지 경로(row["path"])의 기준 디렉토리

# ---- 2) 고정 train/validation split ----
# 매 실험마다 동일한 validation을 쓰기 위해 split을 파일로 고정
sys.path.insert(0, str(REPO_ROOT / "eda"))
from split_validation import ensure_split
ensure_split(RAW_DATA_DIR, SPLIT_DIR)

# ---- 3) CSV 로드 ----
train_df = pd.read_csv(SPLIT_DIR / "train.csv")
valid_df = pd.read_csv(SPLIT_DIR / "validation.csv")
test_df = pd.read_csv(RAW_DATA_DIR / "test.csv")
sample_sub = pd.read_csv(RAW_DATA_DIR / "sample_submission.csv")

# train/valid 간 id 중복이 없는지 확인 (데이터 누수 방지)
assert not set(train_df["id"]) & set(valid_df["id"])

# 학습 데이터 셔플 (SEED 고정으로 재현 가능)
train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("raw  :", RAW_DATA_DIR)
print("split:", SPLIT_DIR)
print(f"train={len(train_df)} | valid={len(valid_df)} | test={len(test_df)}")

## 5. 모델 로드 & LoRA 부착
- 4bit 양자화된 베이스 모델은 고정(freeze)하고, LoRA 어댑터만 학습합니다.
- 이미지 해상도 범위를 processor에 설정합니다.
- 정답 후보 `a/b/c/d`의 토큰 ID를 구해둡니다 (학습 라벨 마스킹, 추론 로짓 추출에 사용).

In [ ]:
# ---- 베이스 모델 + processor 로드 ----
model, processor = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,                       # 4bit 양자화 로드 (VRAM 절약)
    use_gradient_checkpointing="unsloth",    # 활성값 재계산으로 메모리 절약 (Unsloth 최적화 버전)
    max_seq_length=MAX_SEQ_LEN,
)

# ---- LoRA 어댑터 부착 ----
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=FINETUNE_VISION,
    finetune_language_layers=FINETUNE_LANGUAGE,
    finetune_attention_modules=FINETUNE_ATTENTION,
    finetune_mlp_modules=FINETUNE_MLP,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",           # bias는 학습하지 않음
    random_state=SEED,     # LoRA 가중치 초기화 시드
)
model.print_trainable_parameters()  # 학습 가능한 파라미터 수 / 비율 출력

# ---- tokenizer & 이미지 해상도 설정 ----
tokenizer = processor.tokenizer
# Qwen-VL 계열 image processor는 size를 "총 픽셀 수" 범위로 해석
processor.image_processor.size = {
    "shortest_edge": int(MIN_PIXELS),  # 최소 총 픽셀 수
    "longest_edge": int(MAX_PIXELS),   # 최대 총 픽셀 수
}

# ---- 정답 후보 토큰 ID ----
# "a","b","c","d" 각각이 단일 토큰인지 검증 (decode 결과가 원문과 일치해야 함)
CHOICE_IDS = [tokenizer.convert_tokens_to_ids(c) for c in CHOICES]
assert all(tokenizer.decode([t]) == c for t, c in zip(CHOICE_IDS, CHOICES))
print("choice token ids:", dict(zip(CHOICES, CHOICE_IDS)))

## 6. 프롬프트 / Dataset / Collator
- `build_messages`: 한 샘플을 chat 형식(system → user[이미지+텍스트] → assistant[정답])으로 변환
- `VQADataset`: DataFrame 행을 학습용 messages로 변환
- `collator`: Unsloth collator로 토큰화한 뒤, **라벨에서 a/b/c/d 토큰만 남기고 나머지는 -100(무시)** 처리

In [ ]:
def load_image(rel_path):
    """CSV의 상대경로로 이미지를 열어 RGB로 변환 (흑백/RGBA 이미지 대응)."""
    return Image.open(RAW_DATA_DIR / rel_path).convert("RGB")


def build_messages(row, answer=None):
    """
    한 샘플을 Qwen chat template 형식의 messages로 변환.

    - answer=None  : 추론용 (assistant 턴 없음 → 모델이 다음 토큰으로 정답을 예측)
    - answer="a".. : 학습용 (assistant 턴에 정답 한 글자 포함)
    """
    prompt = (
        f"{row['question']}\n"
        f"(a) {row['a']}\n"
        f"(b) {row['b']}\n"
        f"(c) {row['c']}\n"
        f"(d) {row['d']}\n\n"
        f"{USER_INSTRUCTION}"
    )
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image", "image": load_image(row["path"])},
            {"type": "text", "text": prompt},
        ]},
    ]
    if answer is not None:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
    return messages


class VQADataset(Dataset):
    """DataFrame → {"messages": [...]} 형태로 반환하는 학습용 Dataset.
    이미지는 __getitem__ 시점에 로드 (메모리 절약)."""

    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        answer = str(row["answer"]).strip().lower()  # 정답 표기 정규화 (" A " → "a")
        return {"messages": build_messages(row, answer=answer)}


train_ds = VQADataset(train_df)

# ---- Unsloth 비전 collator ----
# chat template 적용 + 이미지 전처리 + 토큰화 + 라벨 생성을 한 번에 수행
base_collator = UnslothVisionDataCollator(
    model,
    processor,
    resize="max",                        # 이미지 리사이즈 기준
    max_seq_length=MAX_SEQ_LEN,
    train_on_responses_only=True,        # user 입력 부분은 라벨 -100, assistant 응답만 학습
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)


def collator(features):
    """
    assistant 응답 중에서도 정답 글자(a/b/c/d) 토큰만 loss에 반영.
    응답 끝의 <|im_end|>, 개행 등은 -100으로 마스킹 → 순수 4지선다 분류 학습이 됨.
    """
    batch = base_collator(features)
    labels = batch["labels"]
    keep = torch.zeros_like(labels, dtype=torch.bool)
    for token_id in CHOICE_IDS:
        keep |= labels.eq(token_id)
    batch["labels"] = torch.where(keep, labels, torch.full_like(labels, -100))
    return batch

## 7. 학습
`SFTTrainer`로 1회 학습 후 LoRA 어댑터만 저장합니다 (베이스 모델 가중치는 저장하지 않음).

`USE_WANDB=True`이면 W&B run을 시작하고, 학습 중 train loss/LR을 `LOGGING_STEPS`마다 W&B로 전송합니다.

In [ ]:
# ---- W&B run 시작 ----
if USE_WANDB:
    wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        config={
            "model": MODEL_NAME,
            "min_pixels": MIN_PIXELS,
            "max_pixels": MAX_PIXELS,
            "max_seq_len": MAX_SEQ_LEN,
            "lora_r": LORA_R,
            "lora_alpha": LORA_ALPHA,
            "finetune_mlp": FINETUNE_MLP,
            "epochs": EPOCHS,
            "learning_rate": LR,
            "batch_size": BATCH_SIZE,
            "gradient_accumulation": GRAD_ACCUM,
            "seed": SEED,
        },
    )

# 학습 모드로 전환 (gradient checkpointing 등 활성화)
FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    data_collator=collator,
    train_dataset=train_ds,
    args=SFTConfig(
        output_dir=str(OUTPUT_DIR / "trainer"),
        # ---- 배치 / epoch ----
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        # ---- 옵티마이저 / 스케줄러 ----
        learning_rate=LR,
        lr_scheduler_type=LR_SCHEDULER,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        optim=OPTIM,
        max_grad_norm=MAX_GRAD_NORM,
        # ---- 정밀도 ----
        bf16=True,   # bfloat16 혼합정밀도 (Ampere 이상)
        tf32=True,
        # ---- 로깅 / 저장 ----
        logging_steps=LOGGING_STEPS,
        save_strategy="no",                          # 중간 체크포인트 저장 안 함
        report_to="wandb" if USE_WANDB else "none",  # train loss/LR을 W&B로 전송
        seed=SEED,
        # ---- 비전 데이터셋 필수 설정 ----
        # messages 안에 PIL 이미지가 있으므로 TRL의 텍스트 전처리를 건너뛰고 collator에서 처리
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=MAX_SEQ_LEN,
        # ---- DataLoader ----
        dataloader_num_workers=4,  # 이미지 로딩 병렬화
    ),
)

trainer.train()

# LoRA 어댑터 + processor 설정 저장 (추후 재로드/앙상블용)
model.save_pretrained(OUTPUT_DIR / "baseline_lora")
processor.save_pretrained(OUTPUT_DIR / "baseline_lora")

## 8. 추론 함수
텍스트를 생성(generate)하지 않고, **assistant 응답 첫 위치의 로짓 중 a/b/c/d 4개만 뽑아 softmax**합니다.
- 항상 4개 중 하나가 선택되므로 형식 오류가 없음
- 확률값을 그대로 얻을 수 있어 이후 앙상블/분석에 활용 가능

In [ ]:
@torch.inference_mode()
def predict_probs(df, batch_size=INFER_BATCH_SIZE, desc="predict"):
    """df의 각 행에 대해 (a, b, c, d) 확률을 반환. shape = (len(df), 4)"""
    probs = np.zeros((len(df), 4), dtype=np.float32)

    # 배치 추론 시 마지막 토큰 위치를 맞추기 위해 왼쪽 패딩 사용
    # (오른쪽 패딩이면 [:, -1]이 패딩 토큰을 가리키게 됨)
    old_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    FastVisionModel.for_inference(model)  # 추론 모드 전환 (Unsloth 추론 최적화)

    try:
        for start in tqdm(range(0, len(df), batch_size), desc=desc):
            batch_df = df.iloc[start:start + batch_size]
            images = [load_image(r["path"]) for _, r in batch_df.iterrows()]

            # add_generation_prompt=True → 끝에 "<|im_start|>assistant\n"이 붙어
            # 다음 토큰 = 정답 글자 위치가 됨
            texts = [
                processor.apply_chat_template(
                    build_messages(r),
                    tokenize=False,
                    add_generation_prompt=True,
                )
                for _, r in batch_df.iterrows()
            ]
            enc = processor(text=texts, images=images, padding=True, return_tensors="pt").to(model.device)

            with torch.autocast("cuda", dtype=torch.bfloat16):
                # logits_to_keep=1: 마지막 위치 로짓만 계산 (메모리 절약)
                # [:, -1, CHOICE_IDS]: 전체 vocab 중 a/b/c/d 4개 로짓만 추출
                logits = model(**enc, logits_to_keep=1).logits[:, -1, CHOICE_IDS]

            probs[start:start + len(batch_df)] = torch.softmax(logits.float(), dim=-1).cpu().numpy()
    finally:
        tokenizer.padding_side = old_side  # 에러가 나도 원래 패딩 방향 복구

    return probs


def accuracy(df, probs):
    """정답 문자(a~d)를 인덱스(0~3)로 바꿔 argmax 예측과 비교한 정확도."""
    gold = df["answer"].str.strip().str.lower().map(CHOICES.index).to_numpy()
    pred = probs.argmax(axis=1)
    return float((pred == gold).mean())

## 9. Validation 평가
validation accuracy를 출력하고, `USE_WANDB=True`이면 W&B에 `validation/accuracy`로 기록합니다.

In [ ]:
valid_probs = predict_probs(valid_df, desc="validation")
valid_acc = accuracy(valid_df, valid_probs)
print(f"validation accuracy: {valid_acc:.4f}")

if USE_WANDB:
    wandb.log({"validation/accuracy": valid_acc})

## 10. Test 추론 & 제출 파일 생성
제출 파일 저장 후 W&B run을 종료합니다 (`wandb.finish()`).

In [ ]:
test_probs = predict_probs(test_df, desc="test")
pred = [CHOICES[i] for i in test_probs.argmax(axis=1)]  # 확률 최대 보기 선택
submission = pd.DataFrame({"id": test_df["id"].values, "answer": pred})

# ---- 제출 형식 검증 ----
assert len(submission) == len(sample_sub)                       # 행 수 일치
assert (submission["id"].values == sample_sub["id"].values).all()  # id 순서 일치
assert submission["answer"].isin(CHOICES).all()                 # 정답 값 유효성

submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)
print("saved:", submission_path)

if USE_WANDB:
    wandb.finish()

submission.head()  # 셀 마지막 줄이어야 출력됨